# Tier4 M0 Verification (Baseline vs Patch)

This notebook verifies M0 kernelization is structural-only by running acceptance on:
1. Baseline pinned commit (detached HEAD)
2. Baseline + patch applied from base64 reconstruction

Required pass conditions:
- `PHASE3D_PASS` in both runs
- strict replay status `ok` in both runs
- `scripts/diff_bundle.py` returns `PASS`

Final marker: `M0_VERIFICATION_PASS`


## 1) Parameters


In [48]:
!rm -rf /content/cholla

In [45]:
REPO_URL = "https://github.com/JinchuLi2002/cholla.git"
BASELINE_SHA = "406dade4561f9212e5cc9ded6aae695219249686"  # TODO: paste pinned baseline commit SHA
PATCH_PATH = "/content/m0.patch"
CONFIG_PATH = "configs/phase3c_acceptance.yaml"
EXPECTED_PATCH_SHA256 = "c8fc71665ee7f8aff757f721a65691af1e3752c285708214dfb949a8936371af"

CLONE_DIR = "/content/cholla"
BUNDLE_ARCHIVE_ROOT = "/content/m0_bundles"
SUMMARY_JSON_PATH = "/content/m0_verification_summary.json"

print({
    "REPO_URL": REPO_URL,
    "BASELINE_SHA": BASELINE_SHA,
    "PATCH_PATH": PATCH_PATH,
    "CONFIG_PATH": CONFIG_PATH,
    "EXPECTED_PATCH_SHA256": EXPECTED_PATCH_SHA256,
})


{'REPO_URL': 'https://github.com/JinchuLi2002/cholla.git', 'BASELINE_SHA': '406dade4561f9212e5cc9ded6aae695219249686', 'PATCH_PATH': '/content/m0.patch', 'CONFIG_PATH': 'configs/phase3c_acceptance.yaml', 'EXPECTED_PATCH_SHA256': 'c8fc71665ee7f8aff757f721a65691af1e3752c285708214dfb949a8936371af'}


## 2) Patch injection (base64)


In [46]:
%%bash
set -euo pipefail

cat > /content/m0.patch.b64 <<'B64_EOF'
...
B64_EOF

base64 -d /content/m0.patch.b64 > /content/m0.patch

python - <<'PY'
import hashlib
from pathlib import Path
p = Path('/content/m0.patch')
data = p.read_bytes()
print({'patch_path': str(p), 'sha256': hashlib.sha256(data).hexdigest(), 'bytes': len(data)})
PY


{'patch_path': '/content/m0.patch', 'sha256': 'c8fc71665ee7f8aff757f721a65691af1e3752c285708214dfb949a8936371af', 'bytes': 41285}


In [47]:
import hashlib
from pathlib import Path

patch_bytes = Path(PATCH_PATH).read_bytes()
PATCH_SHA256_ACTUAL = hashlib.sha256(patch_bytes).hexdigest()

assert PATCH_SHA256_ACTUAL == EXPECTED_PATCH_SHA256, (
    f"patch sha256 mismatch: expected {EXPECTED_PATCH_SHA256}, got {PATCH_SHA256_ACTUAL}"
)
print("patch sha256 ok")


patch sha256 ok


## 3) Fresh clone + detached checkout at BASELINE_SHA


In [49]:
import os
import shutil
import subprocess
from pathlib import Path

if not BASELINE_SHA.strip():
    raise RuntimeError("BASELINE_SHA is empty. Paste the pinned baseline SHA in the parameter cell.")

os.chdir('/content')
repo = Path(CLONE_DIR)
if repo.exists():
    shutil.rmtree(repo)

subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', BASELINE_SHA], check=True)

head_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
if head_sha != BASELINE_SHA:
    raise RuntimeError(f"Detached HEAD mismatch: expected {BASELINE_SHA}, got {head_sha}")

sym = subprocess.run(['git', '-C', str(repo), 'symbolic-ref', '-q', 'HEAD'], text=True, capture_output=True)
if sym.returncode == 0:
    raise RuntimeError(f"HEAD is not detached: {sym.stdout.strip()}")

print({'repo': str(repo), 'head_sha': head_sha, 'detached': True})


{'repo': '/content/cholla', 'head_sha': '406dade4561f9212e5cc9ded6aae695219249686', 'detached': True}


## 4) Safety guard (no push/pull workflow)


In [50]:
import json
from pathlib import Path

# Guard source text when notebook file is available on disk.
notebook_candidates = [
    Path('notebooks/tier4_M0_verification_colab.ipynb'),
    Path('/content/cholla/notebooks/tier4_M0_verification_colab.ipynb'),
]
bad_push = 'git' + ' ' + 'push'
bad_pull = 'git' + ' ' + 'pull'

checked_any = False
for nb_path in notebook_candidates:
    if not nb_path.is_file():
        continue
    checked_any = True
    payload = json.loads(nb_path.read_text(encoding='utf-8'))
    text = "\n".join(''.join(cell.get('source', [])) for cell in payload.get('cells', []))
    assert bad_push not in text, f'Forbidden command found: {bad_push}'
    assert bad_pull not in text, f'Forbidden command found: {bad_pull}'

if not checked_any:
    print('Notebook file not found on disk; runtime command guard still enforced by explicit command set.')

print('safety guard ok')


Notebook file not found on disk; runtime command guard still enforced by explicit command set.
safety guard ok


## 5) Install dependencies


In [51]:
%%bash
set -euo pipefail
cd /content/cholla
python3 -m pip install -q --upgrade pip
python3 -m pip install -q -e .
python3 -m pip install -q pyyaml openai h5py numpy
python3 -m pip --version


pip 26.0.1 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


## 6) Discover toolchain paths


In [52]:
%cd /content/cholla

import glob
import json
import shutil
from pathlib import Path


def _first_match(patterns):
    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return Path(matches[0]).resolve()
    return None


def _reset_link(path: Path, target: Path) -> None:
    if path.exists() or path.is_symlink():
        if path.is_symlink() or path.is_file():
            path.unlink()
        else:
            shutil.rmtree(path)
    path.symlink_to(target)


shim_root = Path('/content/cholla/.colab_toolchain')
shim_root.mkdir(parents=True, exist_ok=True)

cuda_header = _first_match([
    '/usr/local/cuda/include/cuda_runtime.h',
    '/usr/local/cuda*/targets/*/include/cuda_runtime.h',
    '/usr/include/cuda_runtime.h',
])
cuda_lib = _first_match([
    '/usr/local/cuda/lib64/libcudart.so',
    '/usr/local/cuda/lib64/libcudart.so.*',
    '/usr/local/cuda*/targets/*/lib/libcudart.so',
    '/usr/local/cuda*/targets/*/lib/libcudart.so.*',
    '/usr/lib/x86_64-linux-gnu/libcudart.so',
    '/usr/lib/x86_64-linux-gnu/libcudart.so.*',
])
if cuda_header is None or cuda_lib is None:
    raise RuntimeError('CUDA toolkit headers/libs not found in this runtime')

cuda_root = shim_root / 'cuda'
(cuda_root / 'include').parent.mkdir(parents=True, exist_ok=True)
_reset_link(cuda_root / 'include', cuda_header.parent)
_reset_link(cuda_root / 'lib64', cuda_lib.parent)

hdf5_header = _first_match([
    '/usr/include/hdf5/serial/hdf5.h',
    '/usr/include/hdf5/openmpi/hdf5.h',
    '/usr/include/hdf5.h',
])
hdf5_lib = _first_match([
    '/usr/lib/x86_64-linux-gnu/hdf5/serial/libhdf5.so',
    '/usr/lib/x86_64-linux-gnu/hdf5/serial/libhdf5_serial.so',
    '/usr/lib/x86_64-linux-gnu/hdf5/openmpi/libhdf5.so',
    '/usr/lib/x86_64-linux-gnu/hdf5/openmpi/libhdf5_openmpi.so',
    '/usr/lib/x86_64-linux-gnu/libhdf5.so',
    '/usr/lib/x86_64-linux-gnu/libhdf5_serial.so',
])
if hdf5_header is None or hdf5_lib is None:
    raise RuntimeError('HDF5 headers/libs not found in this runtime')

hdf5_root = shim_root / 'hdf5'
(hdf5_root / 'include').parent.mkdir(parents=True, exist_ok=True)
_reset_link(hdf5_root / 'include', hdf5_header.parent)

hdf5_lib_dir = hdf5_root / 'lib'
hdf5_lib_dir.mkdir(parents=True, exist_ok=True)
hdf5_link = hdf5_lib_dir / 'libhdf5.so'
if hdf5_link.exists() or hdf5_link.is_symlink():
    hdf5_link.unlink()
hdf5_link.symlink_to(hdf5_lib)

mpi_root = Path('/usr/lib/x86_64-linux-gnu/openmpi')
if not mpi_root.exists():
    raise RuntimeError(f'MPI root missing: {mpi_root}')

env_payload = {
    'CUDA_ROOT': str(cuda_root),
    'HDF5_ROOT': str(hdf5_root),
    'MPI_ROOT': str(mpi_root),
}

artifacts = Path('/content/cholla/artifacts')
artifacts.mkdir(parents=True, exist_ok=True)
(artifacts / 'env_colab.json').write_text(
    json.dumps(env_payload, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)
print(json.dumps(env_payload, sort_keys=True))


/content/cholla
{"CUDA_ROOT": "/content/cholla/.colab_toolchain/cuda", "HDF5_ROOT": "/content/cholla/.colab_toolchain/hdf5", "MPI_ROOT": "/usr/lib/x86_64-linux-gnu/openmpi"}


## 7) Build Cholla binary


In [53]:
%%bash
set -euo pipefail
cd /content/cholla

eval "$(python3 - <<'PY'
import json
import shlex
from pathlib import Path
payload = json.loads(Path('artifacts/env_colab.json').read_text(encoding='utf-8'))
for key in ('CUDA_ROOT', 'HDF5_ROOT', 'MPI_ROOT'):
    print(f'export {key}={shlex.quote(payload[key])}')
PY
)"

build_log=/tmp/cholla_m0_build.log
env CHOLLA_MACHINE=github CUDA_ROOT="$CUDA_ROOT" HDF5_ROOT="$HDF5_ROOT" MPI_ROOT="$MPI_ROOT"           make TYPE=cosmology -j2 >"$build_log" 2>&1 || (tail -n 200 "$build_log" && exit 1)

tail -n 30 "$build_log"
test -f bin/cholla.cosmology.github
ls -lh bin/cholla.cosmology.github


nvcc -g -O3 -std=c++17 -DMPI_CHOLLA -DPRECISION=2 -DHLLC -DSIMPLE -DPPMP -DDENSITY_FLOOR -DTEMPERATURE_FLOOR -DOUTPUT -DHDF5  -DGRAVITY -DPARIS -DGRAVITY_GPU -DGRAVITY_5_POINTS_GRADIENT -DPARALLEL_OMP -DN_OMP_THREADS=7  -DPARTICLES -DPARTICLES_GPU -DPARTICLE_IDS -DSINGLE_PARTICLE_MASS -DPARALLEL_OMP -DN_OMP_THREADS=7 -DCOSMOLOGY -DAVERAGE_SLOW_CELLS -DDE -DPRINT_INITIAL_STATS -DN_OUTPUT_COMPLETE=1 -DPARIS_5PT -DGIT_HASH='"406dade4561f9212e5cc9ded6aae695219249686"' -DMACRO_FLAGS='"-DMPI_CHOLLA -DPRECISION=2 -DHLLC -DSIMPLE -DPPMP -DDENSITY_FLOOR -DTEMPERATURE_FLOOR -DOUTPUT -DHDF5  -DGRAVITY -DPARIS -DGRAVITY_GPU -DGRAVITY_5_POINTS_GRADIENT -DPARALLEL_OMP -DN_OMP_THREADS=7  -DPARTICLES -DPARTICLES_GPU -DPARTICLE_IDS -DSINGLE_PARTICLE_MASS -DPARALLEL_OMP -DN_OMP_THREADS=7 -DCOSMOLOGY -DAVERAGE_SLOW_CELLS -DDE -DPRINT_INITIAL_STATS -DN_OUTPUT_COMPLETE=1 -DPARIS_5PT -DGIT_HASH='"406dade4561f9212e5cc9ded6aae695219249686"'"' -Isrc -I/content/cholla/.colab_toolchain/hdf5/include -I/usr/lib/x8

## 8) Validate config + LM environment


In [54]:
%cd /content/cholla

import json
import os
from getpass import getpass
from pathlib import Path

import yaml

cfg_path = Path(CONFIG_PATH)
if not cfg_path.is_file():
    raise FileNotFoundError(f"Missing config: {cfg_path}")

cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
if not isinstance(cfg, dict):
    raise RuntimeError(f"Config must decode to mapping: {cfg_path}")

lm_cfg = cfg.get('lm', {}) if isinstance(cfg.get('lm'), dict) else {}
provided_secrets = []
if bool(lm_cfg.get('enabled', True)):
    provider = str(lm_cfg.get('provider', 'openai')).strip().lower() or 'openai'
    if provider == 'azure':
        req = [
            str(lm_cfg.get('azure_env_var_names', {}).get('api_key', 'AZURE_OPENAI_API_KEY')),
            str(lm_cfg.get('azure_env_var_names', {}).get('endpoint', 'AZURE_OPENAI_ENDPOINT')),
            str(lm_cfg.get('azure_env_var_names', {}).get('api_version', 'AZURE_OPENAI_API_VERSION')),
        ]
    else:
        req = [str(lm_cfg.get('openai_env_var_names', {}).get('api_key', 'OPENAI_API_KEY'))]

    missing = [name for name in req if not os.environ.get(name, '').strip()]
    for name in missing:
        secret = getpass(f'Enter value for {name}: ').strip()
        if secret:
            os.environ[name] = secret
            provided_secrets.append(name)

    still_missing = [name for name in req if not os.environ.get(name, '').strip()]
    if still_missing:
        raise RuntimeError(f"Missing LM env vars for provider={provider}: {still_missing}")

print(json.dumps({
    'config_path': str(cfg_path.resolve()),
    'lm_enabled': bool(lm_cfg.get('enabled', True)),
    'tool_backend': cfg.get('tool_backend'),
    'provided_secrets': provided_secrets,
}, sort_keys=True))


/content/cholla
{"config_path": "/content/cholla/configs/phase3c_acceptance.yaml", "lm_enabled": true, "provided_secrets": [], "tool_backend": "real"}


## 9) Helper: acceptance runner + bundle detection


In [55]:
%cd /content/cholla
import yaml, shutil
from pathlib import Path

src = Path(CONFIG_PATH)  # currently configs/phase3c_acceptance.yaml
cfg = yaml.safe_load(src.read_text(encoding="utf-8"))

# Structural verification mode (stable, no real binary crash)
cfg.setdefault("lm", {})["enabled"] = False
cfg["tool_backend"] = "mock"

# Keep IDs short and deterministic
cfg["controller_run_id"] = "m0"
cfg["experiment_id"] = "m0"
cfg["history_path"] = "agent/history/m0_verify.jsonl"
cfg["out_root"] = "r"

ctrl = cfg.setdefault("controller", {})
ctrl["max_iterations"] = 1
ctrl["min_success_iters"] = 1
ctrl["max_failures"] = 1
budgets = ctrl.setdefault("budgets", {})
budgets["max_tool_calls"] = 4
budgets["walltime_budget_sec"] = 120.0

effective = Path("/content/phase3c_m0_verify.yaml")
effective.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
CONFIG_PATH = str(effective)

# Clean old outputs for this ID
shutil.rmtree(Path("agent/experiments") / "m0", ignore_errors=True)
shutil.rmtree(Path("r") / "m0", ignore_errors=True)
Path("agent/history/m0_verify.jsonl").unlink(missing_ok=True)

print("CONFIG_PATH =", CONFIG_PATH)


/content/cholla
CONFIG_PATH = /content/phase3c_m0_verify.yaml


In [56]:
%cd /content/cholla

import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path(CLONE_DIR).resolve()
BUNDLES_ROOT = REPO / 'agent' / 'experiments'
ARCHIVE_ROOT = Path(BUNDLE_ARCHIVE_ROOT)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)

ACCEPT_CMD = [
    sys.executable,
    '-m',
    'agent.controller.run_phase3c',
    '--config',
    CONFIG_PATH,
    '--acceptance_replay_strict',
]

def _bundle_candidates() -> list[Path]:
    out = [path.resolve() for path in BUNDLES_ROOT.glob('*/controller') if path.is_dir()]
    out.sort(key=lambda p: p.stat().st_mtime)
    return out

def _json_payloads(stdout_text: str) -> list[dict]:
    payloads = []
    for raw in stdout_text.splitlines():
        line = raw.strip()
        if not (line.startswith('{') and line.endswith('}')):
            continue
        try:
            parsed = json.loads(line)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            payloads.append(parsed)
    return payloads

def _bundle_from_payloads(payloads: list[dict]) -> Path | None:
    for payload in reversed(payloads):
        raw = payload.get('experiment_bundle_path')
        if isinstance(raw, str) and raw.strip():
            path = Path(raw.strip())
            if not path.is_absolute():
                path = (REPO / path).resolve()
            return path.resolve()
    return None

def run_acceptance(label: str) -> dict:
    before = set(_bundle_candidates())
    proc = subprocess.run(ACCEPT_CMD, cwd=REPO, text=True, capture_output=True)

    print(f'=== {label} CMD ===')
    print(' '.join(ACCEPT_CMD))
    print(f'=== {label} STDOUT ===')
    if proc.stdout.strip():
        print(proc.stdout)
    print(f'=== {label} STDERR ===')
    if proc.stderr.strip():
        print(proc.stderr)

    if proc.returncode != 0:
        raise RuntimeError(f'{label}: acceptance command failed with exit={proc.returncode}')
    if 'PHASE3D_PASS' not in proc.stdout:
        raise RuntimeError(f'{label}: acceptance output missing PHASE3D_PASS marker')

    payloads = _json_payloads(proc.stdout)
    if not payloads:
        raise RuntimeError(f'{label}: no JSON payload emitted by acceptance command')

    result_payload = payloads[-1]
    strict = result_payload.get('strict_replay')
    if not isinstance(strict, dict) or strict.get('status') != 'ok':
        raise RuntimeError(f'{label}: strict replay status is not ok: {strict!r}')

    selected = _bundle_from_payloads(payloads)
    after = _bundle_candidates()
    new_candidates = [path for path in after if path not in before]

    if selected is None or not selected.is_dir():
        if new_candidates:
            selected = new_candidates[-1]
        elif after:
            selected = after[-1]
        else:
            raise RuntimeError(f'{label}: unable to locate bundle path after acceptance run')

    archive_path = ARCHIVE_ROOT / f'{label}_bundle'
    if archive_path.exists():
        shutil.rmtree(archive_path)
    shutil.copytree(selected, archive_path)

    return {
        'label': label,
        'cmd': ACCEPT_CMD,
        'stdout': proc.stdout,
        'stderr': proc.stderr,
        'result_payload': result_payload,
        'bundle_source': str(selected.resolve()),
        'bundle_copy': str(archive_path.resolve()),
    }

print({'repo': str(REPO), 'accept_cmd': ACCEPT_CMD})


/content/cholla
{'repo': '/content/cholla', 'accept_cmd': ['/usr/bin/python3', '-m', 'agent.controller.run_phase3c', '--config', '/content/phase3c_m0_verify.yaml', '--acceptance_replay_strict']}


## 10) Run baseline acceptance (BUNDLE_A)


In [57]:
baseline_run = run_acceptance('baseline')
BUNDLE_A = baseline_run['bundle_copy']
print(json.dumps({
    'BUNDLE_A': BUNDLE_A,
    'bundle_source': baseline_run['bundle_source'],
    'strict_replay': baseline_run['result_payload'].get('strict_replay'),
}, indent=2, sort_keys=True))


=== baseline CMD ===
/usr/bin/python3 -m agent.controller.run_phase3c --config /content/phase3c_m0_verify.yaml --acceptance_replay_strict
=== baseline STDOUT ===
{"acceptance_replay_strict": true, "controller_run_id": "m0", "experiment_bundle_path": "/content/cholla/agent/experiments/m0/controller", "experiment_id": "m0", "failed_iterations": 0, "history_path": "/content/cholla/agent/history/m0_verify.jsonl", "iterations_completed": 1, "max_failures": 1, "min_success_iters": 1, "require_live_lm": false, "run_dir": "/content/cholla/r/m0", "strict_replay": {"error": null, "experiment_id": "m0", "iterations_checked": 1, "metrics_checked": 1, "params_checked": 1, "replay_check": null, "replay_mode": "strict", "run_ids_checked": 1, "status": "ok"}, "successful_iterations": 1, "termination_reason": "max_iterations_reached", "tool_calls_used": 3}
PHASE3D_PASS

=== baseline STDERR ===
{
  "BUNDLE_A": "/content/m0_bundles/baseline_bundle",
  "bundle_source": "/content/cholla/agent/experiments/m

In [58]:
%cd /content/cholla
import json
from pathlib import Path

bundles = sorted(Path("agent/experiments").glob("*/controller"), key=lambda p: p.stat().st_mtime)
if not bundles:
    raise SystemExit("No controller bundles found")
b = bundles[-1]
print("bundle:", b)

rows = [json.loads(line) for line in (b / "history_controller.jsonl").read_text().splitlines() if line.strip()]
iters = [r for r in rows if r.get("record_type") == "iteration"]
print("iteration_status:", iters[-1].get("status") if iters else None)
print("iteration_error:", iters[-1].get("error") if iters else None)
print("termination_reason:", rows[-1].get("termination_reason"))


/content/cholla
bundle: agent/experiments/m0/controller
iteration_status: success
iteration_error: None
termination_reason: max_iterations_reached


In [59]:
%cd /content/cholla
import json
from pathlib import Path

b = sorted(Path("agent/experiments").glob("*/controller"), key=lambda p: p.stat().st_mtime)[-1]
rows = [json.loads(x) for x in (b / "history_controller.jsonl").read_text().splitlines() if x.strip()]
it = [r for r in rows if r.get("record_type") == "iteration"][-1]

run_call_path = Path(it["artifact_paths"]["run_cholla_call_path"])
run_call = json.loads(run_call_path.read_text())
print("run_call_path:", run_call_path)
print("stdout_log:", run_call["result"].get("stdout_log_path"))
print("stderr_log:", run_call["result"].get("stderr_log_path"))
print("tool_log:", run_call["result"].get("tool_log_path"))

run_log = Path(run_call["result"]["run_dir"]) / "run.log"
print("\n=== run.log tail ===")
print("\n".join(run_log.read_text(errors="replace").splitlines()[-120:]))


/content/cholla
run_call_path: /content/cholla/r/m0/iter_0/artifacts/tool_01_run_cholla.json
stdout_log: None
stderr_log: None
tool_log: None

=== run.log tail ===
mock backend: synthetic run


## 11) Apply reconstructed patch


In [60]:
%%bash
set -euo pipefail
cd /content/cholla
git apply --whitespace=nowarn /content/m0.patch


## 12) Run post-patch acceptance (BUNDLE_B)


In [61]:
post_run = run_acceptance('post_patch')
BUNDLE_B = post_run['bundle_copy']
print(json.dumps({
    'BUNDLE_B': BUNDLE_B,
    'bundle_source': post_run['bundle_source'],
    'strict_replay': post_run['result_payload'].get('strict_replay'),
}, indent=2, sort_keys=True))


=== post_patch CMD ===
/usr/bin/python3 -m agent.controller.run_phase3c --config /content/phase3c_m0_verify.yaml --acceptance_replay_strict
=== post_patch STDOUT ===
{"acceptance_replay_strict": true, "controller_run_id": "m0", "experiment_bundle_path": "/content/cholla/agent/experiments/m0/controller", "experiment_id": "m0", "failed_iterations": 0, "history_path": "/content/cholla/agent/history/m0_verify.jsonl", "iterations_completed": 1, "max_failures": 1, "min_success_iters": 1, "require_live_lm": false, "run_dir": "/content/cholla/r/m0", "strict_replay": {"error": null, "experiment_id": "m0", "iterations_checked": 1, "metrics_checked": 1, "params_checked": 1, "replay_check": null, "replay_mode": "strict", "run_ids_checked": 1, "status": "ok"}, "successful_iterations": 1, "termination_reason": "max_iterations_reached", "tool_calls_used": 3}
PHASE3D_PASS

=== post_patch STDERR ===
{
  "BUNDLE_B": "/content/m0_bundles/post_patch_bundle",
  "bundle_source": "/content/cholla/agent/exper

## 13) Deterministic bundle diff (must PASS)


In [62]:
import subprocess
import sys

DIFF_CMD = [
    sys.executable,
    'scripts/diff_bundle.py',
    '--a',
    BUNDLE_A,
    '--b',
    BUNDLE_B,
]

proc = subprocess.run(DIFF_CMD, cwd=CLONE_DIR, text=True, capture_output=True)
DIFF_STDOUT = proc.stdout
DIFF_STDERR = proc.stderr
DIFF_EXIT = proc.returncode

print('=== DIFF CMD ===')
print(' '.join(DIFF_CMD))
print('=== DIFF STDOUT ===')
print(DIFF_STDOUT)
if DIFF_STDERR.strip():
    print('=== DIFF STDERR ===')
    print(DIFF_STDERR)

if DIFF_EXIT != 0:
    raise RuntimeError(f'diff_bundle failed with exit={DIFF_EXIT}')
if not DIFF_STDOUT.strip().startswith('PASS'):
    raise RuntimeError('diff_bundle did not report PASS')


=== DIFF CMD ===
/usr/bin/python3 scripts/diff_bundle.py --a /content/m0_bundles/baseline_bundle --b /content/m0_bundles/post_patch_bundle
=== DIFF STDOUT ===
PASS



## 14) Write summary JSON + final marker


In [63]:
import json
import subprocess
from pathlib import Path

head_sha = subprocess.check_output(['git', '-C', CLONE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
summary = {
    'repo_url': REPO_URL,
    'baseline_sha_param': BASELINE_SHA,
    'checked_out_head_sha': head_sha,
    'patch_path': PATCH_PATH,
    'patch_sha256': PATCH_SHA256_ACTUAL,
    'expected_patch_sha256': EXPECTED_PATCH_SHA256,
    'config_path': CONFIG_PATH,
    'acceptance_command': 'python -m agent.controller.run_phase3c --config configs/phase3c_acceptance.yaml --acceptance_replay_strict',
    'bundle_a': BUNDLE_A,
    'bundle_b': BUNDLE_B,
    'baseline_bundle_source': baseline_run['bundle_source'],
    'post_patch_bundle_source': post_run['bundle_source'],
    'diff_command': ' '.join(DIFF_CMD),
    'diff_exit_code': DIFF_EXIT,
    'diff_stdout': DIFF_STDOUT,
    'diff_stderr': DIFF_STDERR,
    'strict_replay_baseline': baseline_run['result_payload'].get('strict_replay'),
    'strict_replay_post_patch': post_run['result_payload'].get('strict_replay'),
    'status': 'pass',
}

summary_path = Path(SUMMARY_JSON_PATH)
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps({'summary_json': str(summary_path), 'status': 'pass'}, sort_keys=True))
print('M0_VERIFICATION_PASS')


{"status": "pass", "summary_json": "/content/m0_verification_summary.json"}
M0_VERIFICATION_PASS


In [64]:
# open the summary for inspection
import json
from pathlib import Path
summary_path = Path(SUMMARY_JSON_PATH)
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, sort_keys=True))
else:
    print(f"Summary file not found: {summary_path}")

{
  "acceptance_command": "python -m agent.controller.run_phase3c --config configs/phase3c_acceptance.yaml --acceptance_replay_strict",
  "baseline_bundle_source": "/content/cholla/agent/experiments/m0/controller",
  "baseline_sha_param": "406dade4561f9212e5cc9ded6aae695219249686",
  "bundle_a": "/content/m0_bundles/baseline_bundle",
  "bundle_b": "/content/m0_bundles/post_patch_bundle",
  "checked_out_head_sha": "406dade4561f9212e5cc9ded6aae695219249686",
  "config_path": "/content/phase3c_m0_verify.yaml",
  "diff_command": "/usr/bin/python3 scripts/diff_bundle.py --a /content/m0_bundles/baseline_bundle --b /content/m0_bundles/post_patch_bundle",
  "diff_exit_code": 0,
  "diff_stderr": "",
  "diff_stdout": "PASS\n",
  "expected_patch_sha256": "c8fc71665ee7f8aff757f721a65691af1e3752c285708214dfb949a8936371af",
  "patch_path": "/content/m0.patch",
  "patch_sha256": "c8fc71665ee7f8aff757f721a65691af1e3752c285708214dfb949a8936371af",
  "post_patch_bundle_source": "/content/cholla/agent/ex